## 1. Data preparation

In [ ]:
# Standard Libraries
import os
import sys
import math
from pathlib import Path
import random
import logging
from typing import List, Dict

# Third-Party Libraries
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torch import nn
from torch.utils.data import Dataset, DataLoader
import dateutil.parser

# Custom Model Imports
from codes.get_normal_attribute import get_normal_attribute
from codes.load_attribute import load_attribute
from codes.pre_data import pre_data
from codes.epsilon_conditional import EpsModel as EpsModel_conditional
from codes.epsilon_unconditional import EpsModel as EpsModel_unconditional
from codes.diffusion import Diffusion
from codes.train import train_conditional
from codes.get_predict_ddim import get_predict

In [ ]:
# Select device (GPU if available, else CPU)
n_gpus = 1
device = torch.device("cuda:0" if (torch.cuda.is_available() and n_gpus > 0) else "cpu")
print(device)

if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

# Set random seed for reproducibility
seed = 42 
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [ ]:
# Define CAMELS path
camels_path = Path('data_example/CAMELS')

# Load the basin list
example_list = np.loadtxt(camels_path / 'example_list.txt', dtype=str, encoding='utf-8')
basin_list = np.loadtxt(camels_path / 'basin_list.txt', dtype=str, encoding='utf-8')

# Load normal attributes from the specified path
path_attribute = camels_path / 'camels_attributes'
normal_attribute = get_normal_attribute(basin_list, path_attribute)

time_step = 365
batch_size = 3200

forcing_sources = {'daymet': ['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)'],
                    'nldas':  ['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)'],
                    'maurer': ['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)']}

# Define date ranges for training
date_ranges = {
    "train_start": "1980-10-01T00:00:00",
    "train_end": "1990-09-30T00:00:00",
    "test_start": "1995-6-26T00:00:00",
    "test_end": "2005-09-30T00:00:00"
}

In [ ]:
# Prepare training data
testloader_list, std = pre_data(time_step, example_list, is_train=False, is_valid=False, date_ranges=date_ranges,
                                 batch_size=batch_size, camels_path=camels_path, forcing_sources=forcing_sources)

In [ ]:
# Iterate through the dataloader list and check data shapes
for dataloader in testloader_list:
    for X, Y in dataloader:
        print(f'X shape: {X.shape}')
        print(f'Y shape: {Y.shape}')
        
        # Ensure the shapes are correct
        assert X.shape[1] == 365, "X shape is incorrect"
        assert Y.shape[1] == 1, "Y shape is incorrect"
        break  # Exit after the first batch
    break  # Exit after the first dataloader

## 2. Instantiate unconditional diffusion model

In [ ]:
n_steps = 1200

# 实例化 eps_model
eps_unconditional = EpsModel_unconditional(static_attr_len=27, 
                                           hidden_size=256,
                                           emb_dim=128, 
                                           future_step=1,
                                           num_layers=1).to(device)

path_unconditional = 'final_models/eps_unconditional_model.pt'
eps_unconditional.load_state_dict(torch.load(path_unconditional))

## 3. Instantiate conditional diffusion model

In [ ]:
eps_conditional = EpsModel_conditional(input_size=15,
                                       static_attr_len=27,
                                       hidden_size=256,
                                       emb_dim=128,
                                       future_step=1,
                                       num_layers=1).to(device)
path_conditional = 'final_models/eps_conditional_model.pt'
eps_conditional.load_state_dict(torch.load(path_conditional))

## 4. Diffusion model sampling

In [ ]:
sampling_steps = 120
num_samples = 100
eta = 0
guide_w = 0
clip_min = 0.0
output_base_dir = 'sample_results/DDIM'

selected_indices = list(range(3))

In [ ]:
# 调用get_predict函数，生成预测和真实值，并计算NSE
nse_results = get_predict(
    eps_conditional,
    eps_unconditional,
    n_steps,
    sampling_steps,
    basin_list,
    selected_indices,
    testloader_list,
    normal_attribute,
    load_attribute,
    batch_size,
    num_samples,
    eta,
    guide_w,
    clip_min,
    output_base_dir,
    device
)
